In [1]:
# ============================================
# TASK 3: SENTIMENT & CORRELATION ANALYSIS
# ============================================

# ============================================
# IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.sentiment.vader import SentimentIntensityAnalyzer
from scipy.stats import pearsonr

import nltk
nltk.download('vader_lexicon')

import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')


# ============================================
# LOAD DATASETS
# ============================================

news_df = pd.read_csv("../data/raw/raw_analyst_ratings.csv")

stock_df = pd.read_csv("../data/raw/NVDA.csv")

print("Datasets loaded successfully")


# ============================================
# DATA CLEANING
# ============================================

news_df['date'] = pd.to_datetime(
    news_df['date'],
    errors='coerce'
)

stock_df['Date'] = pd.to_datetime(
    stock_df['Date']
)

news_df.dropna(subset=['headline'], inplace=True)

stock_df.fillna(method='ffill', inplace=True)

print("Data cleaning completed")


# ============================================
# FILTER FOR NVIDIA NEWS
# ============================================

news_df = news_df[
    news_df['stock'] == 'NVDA'
]

print("Filtered NVIDIA news")


# ============================================
# DATE ALIGNMENT
# ============================================

news_df['date_only'] = news_df['date'].dt.date

stock_df['date_only'] = stock_df['Date'].dt.date

print("Date alignment completed")


# ============================================
# SENTIMENT ANALYSIS USING VADER
# ============================================

sia = SentimentIntensityAnalyzer()

news_df['sentiment_score'] = news_df[
    'headline'
].apply(
    lambda x: sia.polarity_scores(str(x))['compound']
)

print("Sentiment scores generated")


# ============================================
# DAILY SENTIMENT AGGREGATION
# ============================================

daily_sentiment = news_df.groupby(
    'date_only'
)['sentiment_score'].mean().reset_index()

print("Daily sentiment aggregation completed")


# ============================================
# DAILY STOCK RETURNS
# ============================================

stock_df['daily_return'] = stock_df[
    'Adj Close'
].pct_change() * 100

print("Daily returns calculated")


# ============================================
# MERGE DATASETS
# ============================================

merged_df = pd.merge(
    daily_sentiment,
    stock_df[['date_only', 'daily_return']],
    on='date_only',
    how='inner'
)

print("Datasets merged successfully")

print(merged_df.head())


# ============================================
# PEARSON CORRELATION
# ============================================

correlation, p_value = pearsonr(
    merged_df['sentiment_score'],
    merged_df['daily_return']
)

print("\nPearson Correlation:", correlation)
print("P-value:", p_value)


# ============================================
# SCATTER PLOT
# ============================================

plt.figure(figsize=(10,6))

sns.scatterplot(
    x='sentiment_score',
    y='daily_return',
    data=merged_df
)

plt.title(
    f"Sentiment vs Daily Return\nCorrelation = {correlation:.2f}"
)

plt.xlabel("Average Daily Sentiment")
plt.ylabel("Daily Return (%)")

plt.show()


# ============================================
# SENTIMENT CATEGORY CLASSIFICATION
# ============================================

def classify_sentiment(score):

    if score > 0.05:
        return 'Positive'

    elif score < -0.05:
        return 'Negative'

    else:
        return 'Neutral'


merged_df['sentiment_category'] = merged_df[
    'sentiment_score'
].apply(classify_sentiment)

print("Sentiment classification completed")


# ============================================
# AVERAGE RETURNS BY CATEGORY
# ============================================

category_returns = merged_df.groupby(
    'sentiment_category'
)['daily_return'].mean()

print("\nAverage Returns by Sentiment Category:")
print(category_returns)


# ============================================
# BAR CHART
# ============================================

plt.figure(figsize=(8,5))

sns.barplot(
    x=category_returns.index,
    y=category_returns.values
)

plt.title(
    "Average Daily Returns by Sentiment Category"
)

plt.xlabel("Sentiment Category")
plt.ylabel("Average Daily Return (%)")

plt.show()


# ============================================
# INTERPRETATION
# ============================================

print("""

TASK 3 INTERPRETATION:

1. VADER sentiment analysis was selected because
   it performs well on short financial headlines.

2. Daily sentiment scores were aggregated and
   compared with stock daily returns.

3. Pearson correlation was used to measure the
   relationship between sentiment and market movement.

4. Positive correlations may suggest that optimistic
   news contributes to positive stock performance.

5. Weak correlations indicate that stock markets are
   also affected by broader economic and market factors.

6. This analysis demonstrates how sentiment data
   can support predictive financial analytics.

""")

ModuleNotFoundError: No module named 'seaborn'